# Estimating Time to Payment for Receivables


As I have examined the specific behavior of two atypical customers, it is time to move on to a more generalized approch in order to estimate time to payment of Credix portfolio.

In [1]:
import sys
sys.path.append('../')
from case.helper_functions import create_duration_table, add_company_info, add_quod_info, group_rare_categories

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from lifelines import WeibullAFTFitter, CoxPHFitter


Loading the enriched durations_table I have made so far, and excluding specific cases

In [2]:
assets = pd.read_parquet('../data/df_assets.parquet')

In [3]:
duration_table = create_duration_table(assets)

In [4]:
company = pd.read_parquet('../data/df_company_data.parquet')
quod = pd.read_parquet('../data/df_quod.parquet')

In [5]:
duration_table = add_company_info(duration_table, company)
duration_table = add_quod_info(duration_table, quod)

In [6]:
duration_table = duration_table.loc[~duration_table.tax_id.isin(['14133781557079', '60259422305974'])]

In [7]:
duration_table.head()

,asset_id,tax_id,face_value,created_at,settled_at,due_date,reference_date,duration_days,paid,late_payment,...,secondary_cnae_array,legal_nature,is_mei,city,state,zipcode,has_company_info,quod_score,presumed_revenue,has_quod_info
1,b947a6c4-6e17-a418-091d-c04952f85047,00925347588806,165.14,2024-12-03,2024-12-22,2024-12-24,2025-03-19,19.0,1,0,...,NULL,NULL,NULL,NULL,NULL,NULL,0,-1.0,-1.0,0
4,3ee1a83eb0ba837d,87924474238085,6739.70,2024-06-24,2024-08-06,2024-08-08,2025-02-03,43.0,1,0,...,NULL,NULL,NULL,NULL,NULL,NULL,0,-1.0,-1.0,0
5,34ff0791-cf60-abe9-61fc-128af36e80fb,79510873311545,93000.00,2024-02-22,2024-04-24,2024-04-26,2024-10-01,62.0,1,0,...,NULL,NULL,NULL,NULL,NULL,NULL,0,-1.0,-1.0,0
6,d0c86ee5-cbfd-fc62-aaa0-6a4c282e4668,32000772773675,1988.80,2024-04-29,2024-09-01,2024-07-29,2024-11-19,125.0,1,1,...,NULL,NULL,NULL,NULL,NULL,NULL,0,-1.0,-1.0,0
15,7d7b7081-e050-b617-97a5-15eafffec192,32000772773675,2555.69,2024-03-11,2024-09-08,2024-09-09,2024-11-19,181.0,1,0,...,NULL,NULL,NULL,NULL,NULL,NULL,0,-1.0,-1.0,0


Now, as the duration does not have sellers with large concentration of contracts, a predictive model for time to payment can be made withou the influence of those with high concentration which will demand specific credit policies for them, as noted before

In [8]:
duration_table.tax_id.value_counts(normalize=True)

tax_id
00925347588806    0.307479
32000772773675    0.213752
87924474238085    0.144633
66270349769454    0.075339
78199505504257    0.040292
                    ...   
99881788189631    0.000003
87558830246197    0.000003
28140959347687    0.000003
92250763491332    0.000003
43983879217108    0.000003
Name: proportion, Length: 640, dtype: float64

---

Based on previous analysis, I will keep up only with the fields of interest for modeling

 * asset_id
 * tax_id
 * duration_days
 * cohort_month
 * paid

And some explanotory variables:
 * has_company_info
 * main_cnae_description
 * state
 * quod_score 

In [9]:
tbl_model = duration_table[[ 'asset_id', 'tax_id', 'cohort_month',  'duration_days', 'paid', 'has_company_info', 'main_cnae_description', 'state', 'quod_score']]

In [10]:
tbl_model.head()

,asset_id,tax_id,cohort_month,duration_days,paid,has_company_info,main_cnae_description,state,quod_score
1,b947a6c4-6e17-a418-091d-c04952f85047,00925347588806,2024-12,19.0,1,0,NULL,NULL,-1.0
4,3ee1a83eb0ba837d,87924474238085,2024-06,43.0,1,0,NULL,NULL,-1.0
5,34ff0791-cf60-abe9-61fc-128af36e80fb,79510873311545,2024-02,62.0,1,0,NULL,NULL,-1.0
6,d0c86ee5-cbfd-fc62-aaa0-6a4c282e4668,32000772773675,2024-04,125.0,1,0,NULL,NULL,-1.0
15,7d7b7081-e050-b617-97a5-15eafffec192,32000772773675,2024-03,181.0,1,0,NULL,NULL,-1.0


## Train and validation of time split

Observing the cohorts of payment, and based on previous analysis of cohort behavior in KM curves, 2021 and 2022 will not be used for model construction, as their behavior is quite different from year where Credix had gain a massive use (2023 onwards).

As such 2023 and 2024 will be set up for model training and 2025 for validation

In [11]:
tbl_model.groupby(['cohort_month'])['paid'].count()

cohort_month
2021-07        5
2021-08        3
2021-09        2
2021-10       14
2021-11       25
2021-12       24
2022-01       13
2022-02        8
2022-03       20
2022-04       13
2022-05       27
2022-06       16
2022-07       17
2022-08       32
2022-09       68
2022-10       41
2022-11      229
2022-12     1018
2023-01     2539
2023-02     2152
2023-03     3578
2023-04     2543
2023-05     3993
2023-06     3650
2023-07     4149
2023-08     5053
2023-09     5772
2023-10     9920
2023-11    11985
2023-12    12367
2024-01    10115
2024-02     8644
2024-03    10841
2024-04    12340
2024-05    11697
2024-06    12327
2024-07    13562
2024-08    20510
2024-09    21940
2024-10    24444
2024-11    20635
2024-12    17438
2025-01    15005
2025-02    12354
2025-03     8210
2025-04     1160
2025-05      321
2025-06      358
2025-07      118
Name: paid, dtype: int64

In [12]:
train = tbl_model.loc[(tbl_model.cohort_month.str.startswith('2023')) | (tbl_model.cohort_month.str.startswith('2024'))]
val = tbl_model.loc[(tbl_model.cohort_month.str.startswith('2025'))]


As we see, validation cohort has a slightly longer time to payment overall

In [13]:
train.duration_days.median()

np.float64(50.0)

In [14]:
val.duration_days.median()

np.float64(28.0)

---

# Encoding

* main_cnae_description
* state 
* quod_score

Encondando fields "main_cnae description" and "state" with relative frequency for each group

In [15]:
train = group_rare_categories(train, 'main_cnae_description')

In [16]:
dict(train.main_cnae_description.value_counts(normalize=True))

{'NULL': np.float64(0.9369889846705314),
 'Comércio a varejo de peças e acessórios novos para veículos automotores': np.float64(0.03521495356749169),
 'Comércio varejista de cosméticos, produtos de perfumaria e de higiene pessoal': np.float64(0.01316843382475396),
 'Comércio atacadista de produtos alimentícios em geral': np.float64(0.007656803889069526),
 'Frigorífico - abate de bovinos': np.float64(0.0054045694980848075),
 'Demais': np.float64(0.001566254550068598)}

In [17]:
freq_dict = {
    'NULL': 0.936989842795314,
    'Comércio a varejo de peças e acessórios novos para veículos automotores': 0.03521495356749169,
    'Comércio varejista de cosméticos, produtos de perfumaria e de higiene pessoal': 0.01316843382475396,
    'Comércio atacadista de produtos alimentícios em geral': 0.007656803889069526,
    'Frigorífico - abate de bovinos': 0.0054045694980848075,
    'Demais': 0.001566254559068598
}

# Criar o mapeamento
train['main_cnae_description_freq'] = train['main_cnae_description'].map(freq_dict)


---

In [18]:
train = group_rare_categories(train, 'state')

In [19]:
dict(train.state.value_counts(normalize=True))

{'NULL': np.float64(0.9369889846705314),
 'São Paulo': np.float64(0.04871249910782969),
 'Minas Gerais': np.float64(0.013794935644781398),
 'Demais': np.float64(0.0005035805768574986)}

In [20]:
freq_dict = {
    'NULL': 0.9369889846705314,
     'São Paulo': 0.04871249910782969,
     'Minas Gerais': 0.013794935644781398,
     'Demais': 0.0005035805768574986}

train['state_freq'] = train['state'].map(freq_dict)

For score quod, use the encoding previously shown in exploratory analysis, giving lower rank to no information

In [21]:
train.quod_score

1        -1.0
4        -1.0
5        -1.0
6        -1.0
15       -1.0
         ... 
980162   -1.0
980163   -1.0
980166   -1.0
980167   -1.0
980176   -1.0
Name: quod_score, Length: 252194, dtype: float64

In [22]:
train['quod_score']  = np.where(train['quod_score'] == -1, 0,
                                    np.where(train['quod_score'] <920, 1, 2))

Define enconding strategy in a function (CNAE, state and QUOD)

In [23]:
def encode_variables(df):
    """Encode variable main_core_cnae_description, state, and score_quod for further use in survival analysis algorithms"""
    
    df = group_rare_categories(df, 'main_cnae_description')
    freq_dict = {
    'NULL': 0.936989842795314,
    'Comércio a varejo de peças e acessórios novos para veículos automotores': 0.03521495356749169,
    'Comércio varejista de cosméticos, produtos de perfumaria e de higiene pessoal': 0.01316843382475396,
    'Comércio atacadista de produtos alimentícios em geral': 0.007656803889069526,
    'Frigorífico - abate de bovinos': 0.0054045694980848075,
    'Demais': 0.001566254559068598}
    df['main_cnae_description_freq'] = df['main_cnae_description'].map(freq_dict)

    df = group_rare_categories(df, 'state')

    freq_dict = {
        'NULL': 0.9369889846705314,
        'São Paulo': 0.04871249910782969,
        'Minas Gerais': 0.013794935644781398,
        'Demais': 0.0005035805768574986}

    df['state_freq'] = df['state'].map(freq_dict)
    
    df['quod_score']  = np.where(df['quod_score'] == -1, 0,
                                    np.where(df['quod_score'] <920, 1, 2))

    return df   


---

# Treinando alguns modelos AFT (não exaustivo)

Training an accelerated failure time using the traditional Weibull distribution, without penalization, we get:

In [24]:
train_treated = train[['duration_days', 'paid', 'main_cnae_description_freq', 'state_freq', 'quod_score', 'has_company_info']]

In [25]:
#needed duto to 0 duration (not allowed)
train_treated['duration_days'] = train_treated['duration_days'] + 1

In [26]:

aft_1 = WeibullAFTFitter()

aft_1.fit(
    train_treated,
    duration_col='duration_days',
    event_col='paid'
)

<lifelines.WeibullAFTFitter: fitted with 252194 total observations, 24091 right-censored observations>

In [27]:
aft_1.summary

coef     exp(coef)  se(coef)  \
param   covariate                                                       
lambda_ has_company_info            36.300920  5.824911e+15  2.856161   
        main_cnae_description_freq  34.134472  6.674410e+14  2.294923   
        quod_score                  -0.084834  9.186645e-01  0.046609   
        state_freq                   6.277403  5.324043e+02  1.026791   
        Intercept                  -33.558613  2.664892e-15  2.883567   
rho_    Intercept                    0.223881  1.250922e+00  0.001459   

                                    coef lower 95%  coef upper 95%  \
param   covariate                                                    
lambda_ has_company_info                 30.702948       41.898892   
        main_cnae_description_freq       29.636505       38.632439   
        quod_score                       -0.176185        0.006517   
        state_freq                        4.264930        8.289876   
        Intercept                       -39.210299      -27.906926   
rho_    Intercept                         0.221020        0.226741   

                                    exp(coef) lower 95%  exp(coef) upper 95%  \
param   covariate                                                              
lambda_ has_company_info                   2.158345e+13         1.572018e+18   
        main_cnae_description_freq         7.429688e+12         5.995910e+16   
        quod_score                         8.384626e-01         1.006538e+00   
        state_freq                         7.115995e+01         3.983342e+03   
        Intercept                          9.358006e-18         7.588852e-13   
rho_    Intercept                          1.247349e+00         1.254505e+00   

                                    cmp to           z             p  \
param   covariate                                                      
lambda_ has_company_info               0.0   12.709692  5.224011e-37   
        main_cnae_description_freq     0.0   14.873905  4.868830e-50   
        quod_score                     0.0   -1.820146  6.873681e-02   
        state_freq                     0.0    6.113614  9.739964e-10   
        Intercept                      0.0  -11.637884  2.644978e-31   
rho_    Intercept                      0.0  153.405778  0.000000e+00   

                                      -log2(p)  
param   covariate                               
lambda_ has_company_info            120.526182  
        main_cnae_description_freq  163.812830  
        quod_score                    3.862773  
        state_freq                   29.935365  
        Intercept                   101.576515  
rho_    Intercept                          inf

On this setup, it can be seen a divergente behavior for quod_score (it was expected to have a positive coefficient), and all others were quite high, specially company_info and cnae variables. Concordance index, shows this initial model only slightly better than random choice

In [28]:
aft_1.concordance_index_

np.float64(0.5385628619330058)

---

In [29]:
aft_2 = WeibullAFTFitter(penalizer=0.01)

aft_2.fit(
    train_treated,
    duration_col='duration_days',
    event_col='paid'
)

<lifelines.WeibullAFTFitter: fitted with 252194 total observations, 24091 right-censored observations>

In [30]:
aft_2.summary

coef  exp(coef)  se(coef)  \
param   covariate                                                   
lambda_ has_company_info           -0.384869   0.680540  0.067414   
        main_cnae_description_freq  0.537574   1.711850  0.072901   
        quod_score                  0.481787   1.618965  0.007524   
        state_freq                  0.395450   1.485053  0.074040   
        Intercept                   3.432001  30.938477  0.068797   
rho_    Intercept                   0.222192   1.248811  0.001458   

                                    coef lower 95%  coef upper 95%  \
param   covariate                                                    
lambda_ has_company_info                 -0.516997       -0.252740   
        main_cnae_description_freq        0.394692        0.680457   
        quod_score                        0.467041        0.496534   
        state_freq                        0.250335        0.540566   
        Intercept                         3.297161        3.566840   
rho_    Intercept                         0.219334        0.225049   

                                    exp(coef) lower 95%  exp(coef) upper 95%  \
param   covariate                                                              
lambda_ has_company_info                       0.596308             0.776670   
        main_cnae_description_freq             1.483927             1.974780   
        quod_score                             1.595267             1.643016   
        state_freq                             1.284456             1.716978   
        Intercept                             27.035782            35.404537   
rho_    Intercept                              1.245248             1.252385   

                                    cmp to           z             p  \
param   covariate                                                      
lambda_ has_company_info               0.0   -5.709045  1.136119e-08   
        main_cnae_description_freq     0.0    7.374075  1.654895e-13   
        quod_score                     0.0   64.035399  0.000000e+00   
        state_freq                     0.0    5.341058  9.240549e-08   
        Intercept                      0.0   49.886011  0.000000e+00   
rho_    Intercept                      0.0  152.399389  0.000000e+00   

                                     -log2(p)  
param   covariate                              
lambda_ has_company_info            26.391311  
        main_cnae_description_freq  42.458325  
        quod_score                        inf  
        state_freq                  23.367446  
        Intercept                         inf  
rho_    Intercept                         inf

In [31]:
aft_2.concordance_index_

np.float64(0.5385573181485191)

---

Repeating the training without quod variable, it was obtained a fairly reasonable model with all covariates having a positive impact

In [32]:
train_treated = train[['duration_days', 'paid', 'main_cnae_description_freq', 'state_freq', 'has_company_info']]

In [33]:
train_treated['duration_days'] = train_treated['duration_days'] + 1

In [34]:
aft_3 = WeibullAFTFitter(penalizer=0.01)

aft_3.fit(
    train_treated,
    duration_col='duration_days',
    event_col='paid'
)

<lifelines.WeibullAFTFitter: fitted with 252194 total observations, 24091 right-censored observations>

In [35]:
aft_3.summary

coef  exp(coef)  se(coef)  \
param   covariate                                                   
lambda_ has_company_info            0.312304   1.366570  0.066462   
        main_cnae_description_freq  0.540373   1.716647  0.073098   
        state_freq                  0.366333   1.442435  0.074428   
        Intercept                   3.453699  31.617124  0.069333   
rho_    Intercept                   0.210785   1.234647  0.001450   

                                    coef lower 95%  coef upper 95%  \
param   covariate                                                    
lambda_ has_company_info                  0.182041        0.442568   
        main_cnae_description_freq        0.397103        0.683643   
        state_freq                        0.220456        0.512209   
        Intercept                         3.317808        3.589589   
rho_    Intercept                         0.207943        0.213627   

                                    exp(coef) lower 95%  exp(coef) upper 95%  \
param   covariate                                                              
lambda_ has_company_info                       1.199663             1.556699   
        main_cnae_description_freq             1.487509             1.981083   
        state_freq                             1.246645             1.668974   
        Intercept                             27.599795            36.219201   
rho_    Intercept                              1.231143             1.238161   

                                    cmp to           z             p  \
param   covariate                                                      
lambda_ has_company_info               0.0    4.698981  2.614628e-06   
        main_cnae_description_freq     0.0    7.392399  1.442031e-13   
        state_freq                     0.0    4.921965  8.567934e-07   
        Intercept                      0.0   49.813081  0.000000e+00   
rho_    Intercept                      0.0  145.365259  0.000000e+00   

                                     -log2(p)  
param   covariate                              
lambda_ has_company_info            18.544963  
        main_cnae_description_freq  42.656963  
        state_freq                  20.154549  
        Intercept                         inf  
rho_    Intercept                         inf

In [36]:
aft_3.concordance_index_

np.float64(0.5385567188653881)

---

In [39]:
val = encode_variables(val)

In [41]:
val.columns

Index(['asset_id', 'tax_id', 'cohort_month', 'duration_days', 'paid',
       'has_company_info', 'main_cnae_description', 'state', 'quod_score',
       'main_cnae_description_freq', 'state_freq'],
      dtype='object')

In [44]:
X_val = val[['has_company_info','main_cnae_description_freq', 'state_freq' ]]

In [45]:
predicted_times = aft_3.predict_median(X_val)

In [46]:
predicted_times

19        54.949597
23        54.949597
44        54.949597
91        54.949597
135       54.949597
            ...    
980025    54.949597
980028    54.949597
980094    33.315439
980096    32.715144
980146    54.949597
Length: 37526, dtype: float64

In [48]:
val['predicted_time_to_pay'] = predicted_times

In [51]:
val.loc[val.paid == 1].duration_days.median()

np.float64(33.0)

In [55]:
val.predicted_time_to_pay.median()

np.float64(54.94959710219951)

# Conclusion

The proposed model at the moment was not able to capture correctly time to payment, accordingly. WIth the given information on data and lack of data for a large portion of the dataset, I have tried to calibrate a  model using a simple parametric strategy, but it overestimates time to payment in a large amount (18 days)

With due time , this methodology could be improved using machine learning methods (non-parametric) and made more exploration. Due to time constraints of this exercise, I wuill share the final estimates of oot time estimates for 2025 in the folder, for analysis.

I was able to construct a pipeline from assets payments to a duration table, enriched with company and bureau data and make some explanotory analysis on payment behavior. After caregul inspection of the assets data, two big sellers were analyzed separated, and a time to payment survival model was attempted to predict the behavior of the remaining population

In [70]:
final = pd.concat([val.loc[val.paid == 1].duration_days, val.predicted_time_to_pay], axis= 1)

In [71]:
final['delta'] = final['predicted_time_to_pay'] - final['duration_days']

In [76]:
final = final.loc[~final.duration_days.isna()]

In [78]:
final.delta.mean()

np.float64(18.328598199630992)

In [79]:
final.to_csv('../data/output/predictions_assets_2025.csv', header=True)